In [1]:
!pip install seaborn matplotlib scanpy numpy igraph harmonypy anndata

In [25]:
import os
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
from matplotlib import pyplot as plt

In [3]:
# -- Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [4]:
# -- Paths
input_path = "/content/drive/MyDrive/endo-immune-atlas/data/processed/GSE179640_integration.h5ad"
output_path_data = "/content/drive/MyDrive/endo-immune-atlas/data/processed"
output_path_figures = "/content/drive/MyDrive/endo-immune-atlas/results/preprocessing"

os.makedirs(output_path_figures, exist_ok=True)

In [5]:
combined = sc.read_h5ad(input_path)
combined

AnnData object with n_obs × n_vars = 94891 × 2000
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet'
    var: 'hemos', 'ribos', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg', 'log1p', 'neighbors', 'patient_id_colors', 'pca', 'scrublet', 'tissue_type_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [6]:
for res in [0.01, 0.02, 0.05]:
    sc.tl.leiden(combined,
                 key_added=f"leiden_res_{res:4.2f}",
                 resolution=res,
                 flavor="igraph",
                 )

In [27]:
sc.pl.umap(
    combined,
    color=["leiden_res_0.01", "leiden_res_0.02", "leiden_res_0.05"],
    legend_loc="on data", title = ["Leiden Resolution 0.01", "Leiden Resolution 0.02", "Leiden Resolution 0.05"], show = False
)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_leiden_res.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [28]:
sc.pl.umap(
    combined,
    color=["leiden_res_0.01", "predicted_doublet", "doublet_score"],
    wspace=0.5,
    size=3,
    palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00"],
    title = ["UMAP Leiden Resolution 0.01", "Predicted Doublets in Clusters", "Doublet Score in Clusters"], show = False
)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_umap_qc_check.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [9]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(
    combined,
    groupby="leiden_res_0.01",
    method="wilcoxon"
)

In [10]:
markers = sc.get.rank_genes_groups_df(
    combined,
    group=None
)

markers = markers[
    (markers["logfoldchanges"] > 1.5) &
    (markers["pvals_adj"] < 0.05)
]

In [11]:
top_markers = (
    markers.sort_values(
        ["group", "logfoldchanges"],
        ascending=[True, False]
    )
    .groupby("group")
    .head(20)
)

/tmp/ipykernel_153527/3163260058.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("group")


In [12]:
for c in markers["group"].unique():
    print(f"\nCluster {c}")
    print(
        markers[markers["group"] == c]
        .sort_values("logfoldchanges", ascending=False)
        .head(10)["names"]
        .tolist()
    )


Cluster 0
['CYBB', 'LILRA5', 'LY86', 'CD86', 'MPEG1', 'FCGR1A', 'S100A12', 'RETN', 'FPR1', 'MS4A7']

Cluster 1
['TRGV10', 'TRAV4', 'KLRC3', 'CD3G', 'CD3E', 'GZMA', 'TRGV2', 'XCL2', 'CD3D', 'LINC02446']

Cluster 2
['DCN', 'FHL5', 'SCN7A', 'KCNE4', 'LMOD1', 'DPT', 'COX4I2', 'TCF21', 'EDNRA', 'PDGFRA']

Cluster 3
['REG1A', 'CLDN3', 'SMIM22', 'LINC01320', 'SLC34A2', 'UCA1', 'WFDC2', 'FXYD3', 'SCGB2A1', 'EPCAM']

Cluster 4
['ECSCR', 'ADGRL4', 'CLEC14A', 'CXorf36', 'CCL14', 'TM4SF18', 'MYCT1', 'VWF', 'CLDN5', 'EMCN']


In [13]:
# -- Marker Gene List
marker_genes = {
    "Myeloid":     ["CD68", "CD14", "CD86", "FCGR1A", "LYZ",
                    "S100A8", "S100A9", "ITGAM", "CSF1R", "MPEG1"],

    "Lymphoid":    ["CD3D", "CD3E", "CD3G", "CD8A", "CD4",
                    "GZMA", "GZMB", "NKG7", "NCAM1", "KLRD1"],

    "Stromal":     ["COL1A1", "COL1A2", "PDGFRA", "DCN", "LUM",
                    "VIM", "ACTA2", "THY1", "POSTN"],

    "Epithelial":  ["EPCAM", "KRT8", "KRT18", "KRT19",
                    "WFDC2", "MUC1", "CLDN3", "CLDN4", "FXYD3"],

    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "EMCN",
                    "ENG", "CLEC14A", "MCAM", "FLT1", "KDR"]
}

In [29]:
# -- Myeloid cluster
sc.pl.umap(combined,
           color = marker_genes.get("Myeloid"),
           legend_loc = "on data",
           frameon = False,
           ncols = 5,
           show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_myeloid_markers.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [30]:
# -- Lymphoid cluster
sc.pl.umap(combined,
           color =  marker_genes.get("Lymphoid"),
           legend_loc = "on data",
           frameon = False,
           ncols = 5,
           show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_lymph_markers.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [31]:
# -- Stromal cluster
sc.pl.umap(combined,
           color = marker_genes.get("Stromal"),
           legend_loc = "on data",
           frameon = False,
           ncols = 3,
           show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_stromal_markers.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [32]:
# -- Epithelial
sc.pl.umap(combined,
           color = marker_genes.get("Epithelial"),
           legend_loc = "on data",
           frameon = False,
           ncols = 3,
           show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_epithelial_markers.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [33]:
# -- Endothelial cluster
sc.pl.umap(combined,
           color = marker_genes.get("Endothelial"),
           legend_loc = "on data",
           frameon = False,
           ncols = 5,
           show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_endo_markers.png"),
    bbox_inches = 'tight',
    dpi = 300
)
plt.close()

In [35]:
combined.obs["cluster_label"] = combined.obs["leiden_res_0.01"].map(
    {
        "0": "Myeloid",
        "1": "Lymphoid",
        "2": "Stromal",
        "3": "Epithelial",
        "4": "Endothelial"
    }
)

sc.pl.dotplot(combined,
              marker_genes,
              groupby="leiden_res_0.01",
              standard_scale="var",
              show = False)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_dot_plot.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [36]:
sc.pl.umap(
    combined,
    color="cluster_label",
    title = "",
    palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00"],
    show = False
)

plt.savefig(
    os.path.join(output_path_figures, "04_total_clustering_umap_labeled.png"),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [37]:
# -- Save clustering object for integration
combined.write_h5ad(os.path.join(output_path_data, "GSE179640_total_clustering.h5ad"))